# 04 - Continuous Time Grid & Lag Structure Audit

This notebook consolidates all 24 monthly 15-minute demand files, audits time continuity across month boundaries, and inspects engineered lag features and target surge deltas ($\Delta y_t$).

In [1]:
import sys
from pathlib import Path

# Resolve project root dynamically regardless of notebook working directory
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root added to sys.path: {PROJECT_ROOT}')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.features import consolidate_demand_matrices, engineer_feature_lags
from src.config import PROCESSED_DATA_DIR

print('Imports successful.')

Project root added to sys.path: C:\Users\rafid\Desktop\Projects\surge-regressor
Imports successful.


### 1. Load and Consolidate 24 Monthly Matrices
Merges individual monthly files from `data/processed/` and drops seam-boundary collisions.

In [3]:
df_consolidated = consolidate_demand_matrices()
print(f'Total Consolidated Records: {len(df_consolidated):,}')

df_consolidated.groupby('venue')['timestamp_15m'].agg(['min', 'max', 'count'])

Total Consolidated Records: 140,352


,min,max,count
venue,,,
barclays,2024-01-01,2025-12-31 23:45:00,70176
citi_field,2024-01-01,2025-12-31 23:45:00,70176


### 2. Time Grid Continuity Audit
Verifies that each step is strictly separated by exactly 15 minutes with 0 gaps or irregular hops.

In [4]:
for venue_name, group in df_consolidated.groupby('venue'):
    time_diffs = group['timestamp_15m'].diff().dropna()
    irregular_steps = (time_diffs != pd.Timedelta('15min')).sum()
    print(f'Venue: {venue_name:<12} | Irregular intervals: {irregular_steps} (Expected: 0)')
    assert irregular_steps == 0, f'Found discontinuous intervals in {venue_name}!'

Venue: barclays     | Irregular intervals: 0 (Expected: 0)
Venue: citi_field   | Irregular intervals: 0 (Expected: 0)


### 3. Feature Lag & Surge Delta Computation
Computes cyclical calendar features (`sin`/`cos`), momentum lags ($t-1, t-2, t-4, t-96, t-672$), rolling statistics, and target delta $\Delta y_t$.

In [5]:
featured_df = engineer_feature_lags(df_consolidated)

# Drop the initial 7-day warmup period where lag_672 (1-week lag) is NaN
ml_ready = featured_df.dropna(subset=['demand_lag_672']).reset_index(drop=True)

display_cols = [
    'timestamp_15m', 'venue', 'weighted_demand',
    'demand_lag_1', 'demand_lag_4', 'demand_lag_672', 'target_delta_y'
]
print(f'ML-Ready matrix shape: {ml_ready.shape}')
ml_ready[display_cols].head(10)

ML-Ready matrix shape: (139008, 22)


,timestamp_15m,venue,weighted_demand,demand_lag_1,demand_lag_4,demand_lag_672,target_delta_y
0,2024-01-08 00:00:00,barclays,46.75,50.50,94.50,253.75,-207.00
1,2024-01-08 00:15:00,barclays,42.50,46.75,92.00,349.00,-306.50
2,2024-01-08 00:30:00,barclays,34.50,42.50,63.00,266.25,-231.75
3,2024-01-08 00:45:00,barclays,33.75,34.50,50.50,287.50,-253.75
4,2024-01-08 01:00:00,barclays,21.75,33.75,46.75,273.00,-251.25
5,2024-01-08 01:15:00,barclays,21.00,21.75,42.50,270.75,-249.75
6,2024-01-08 01:30:00,barclays,15.75,21.00,34.50,273.00,-257.25
7,2024-01-08 01:45:00,barclays,23.25,15.75,33.75,266.75,-243.50
8,2024-01-08 02:00:00,barclays,14.00,23.25,21.75,248.00,-234.00
9,2024-01-08 02:15:00,barclays,16.00,14.00,21.00,245.75,-229.75


### 4. Summary Statistics for Target Surge Delta ($\Delta y_t$)

In [6]:
ml_ready.groupby('venue')['target_delta_y'].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
venue,,,,,,,,
barclays,69504.0,0.00,23.57,-306.5,-8.50,0.25,9.00,289.0
citi_field,69504.0,0.07,14.76,-174.5,-5.25,0.00,5.25,174.0
